# Альфа-банк — Кредиты Наличными: Ranking NDCG@5

**Постановка:** в каждой заявке (`request_id`) есть ~22 оффера, клиент выбирает **ровно один** (`is_deal=1`). Нужно отранжировать офферы внутри заявки по вероятности выбора.

**Метрика:** NDCG@5

**План:**
1. Загрузка и препроцессинг (object → float, dedup features)
2. Join train/test с features по `(app_id, date_part)`
3. Listwise feature engineering (внутри `request_id`)
4. CatBoost YetiRank / LightGBM lambdarank
5. GroupKFold по `request_id`, метрика — NDCG@5
6. Submission в формате commit.csv


In [4]:
# === Установка ===
!pip install -q lightgbm catboost xgboost optuna pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 1.7 MB/s eta 0:00:0000:010m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 40.3 MB/s eta 0:00:00


In [13]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score
import gc, os, warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [15]:
# Ячейка 2: скачиваем данные с Яндекс.Диска
import requests

YANDEX_LINK = "./data"

def download_yadisk(public_link, save_dir="./data"):
    os.makedirs(save_dir, exist_ok=True)
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources"
    
    # Получаем список файлов в папке
    r = requests.get(api_url, params={"public_key": public_link, "limit": 100})
    items = r.json().get("_embedded", {}).get("items", [])
    
    for item in items:
        name = item["name"]
        # Получаем прямую ссылку на скачивание
        dl = requests.get(
            "https://cloud-api.yandex.net/v1/disk/public/resources/download",
            params={"public_key": public_link, "path": "/" + name}
        ).json()
        href = dl["href"]
        print(f"Качаю {name}...")
        with requests.get(href, stream=True) as resp:
            with open(f"{save_dir}/{name}", "wb") as f:
                for chunk in resp.iter_content(8192):
                    f.write(chunk)
    print("Готово")

download_yadisk(YANDEX_LINK)
!ls -lh ./data/

Качаю commit.csv...
Качаю feature_description.csv...
Качаю features_small.pq...
Качаю model_credit_start.ipynb...
Качаю test_dataset_small.pq...
Качаю train_dataset_small.pq...
Готово
total 119M
-rw-r--r-- 1 root root  16M May 21 19:22 commit.csv
-rw-r--r-- 1 root root 2.2K May 21 19:22 feature_description.csv
-rw-r--r-- 1 root root  27M May 21 19:22 features_small.pq
-rw-r--r-- 1 root root 7.4K May 21 19:22 model_credit_start.ipynb
-rw-r--r-- 1 root root  32M May 21 19:22 test_dataset_small.pq
-rw-r--r-- 1 root root  45M May 21 19:22 train_dataset_small.pq


In [16]:
!ls

data  sample_data


In [17]:
DATA_DIR = './data'
# === Загрузка ===
train = pd.read_parquet(f'{DATA_DIR}/train_dataset_small.pq')
test  = pd.read_parquet(f'{DATA_DIR}/test_dataset_small.pq')
feats = pd.read_parquet(f'{DATA_DIR}/features_small.pq')

print('train:', train.shape, '| test:', test.shape, '| feats:', feats.shape)
print('positive rate в train:', train['is_deal'].mean())
print('Сделок на заявку:', train.groupby('request_id')['is_deal'].sum().value_counts())

train: (845110, 23) | test: (936883, 22) | feats: (76583, 280)
positive rate в train: 0.04483558353350452
Сделок на заявку: is_deal
1    37891
Name: count, dtype: int64


In [18]:
# === Препроцессинг: строковые числа -> float ===
# rate, eva, eva_perc, ncl сохранены как Hive Decimal -> строки
NUMERIC_FROM_STR = ['rate', 'eva', 'eva_perc', 'ncl']

for df in (train, test):
    for col in NUMERIC_FROM_STR:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

print('После конвертации:')
print(train[NUMERIC_FROM_STR].dtypes)
print(train[NUMERIC_FROM_STR].describe())

После конвертации:
rate        float32
eva         float32
eva_perc    float32
ncl         float32
dtype: object
                rate            eva       eva_perc            ncl
count  845110.000000  845110.000000  845110.000000  845110.000000
mean       30.823095   23392.009766       0.048581       0.054235
std         6.053184   23883.416016       0.025801       0.048514
min        18.990000       5.182544       0.000000       0.000001
25%        24.990000    7484.539795       0.029765       0.016700
50%        29.990000   15939.035156       0.045514       0.038753
75%        35.990002   31092.375977       0.064287       0.080104
max        42.990002  640743.187500       0.202235       0.331647


In [19]:
# === Категориальные строки -> codes ===
CAT_STR = ['verif_compl', 'offer_type', 'risk_level_map', 
           'verif_need', 'need_2ndfl', 'channel', 'basket_name']

# Конкатенируем train+test для согласованного маппинга
combined_cat_maps = {}
for col in CAT_STR:
    all_vals = pd.concat([train[col], test[col]], axis=0).fillna('NA').astype(str)
    cats = pd.Categorical(all_vals).categories
    combined_cat_maps[col] = {v: i for i, v in enumerate(cats)}
    train[col] = train[col].fillna('NA').astype(str).map(combined_cat_maps[col]).astype('int32')
    test[col]  = test[col].fillna('NA').astype(str).map(combined_cat_maps[col]).astype('int32')

print('Кардинальности cat-фичей:')
for col in CAT_STR:
    print(f'  {col}: {len(combined_cat_maps[col])}')

Кардинальности cat-фичей:
  verif_compl: 2
  offer_type: 2
  risk_level_map: 4
  verif_need: 2
  need_2ndfl: 3
  channel: 2
  basket_name: 13


In [20]:
# === Дедуп features по (app_id, date_part) ===
# В feats были 148 дубликатов по app_id. Берём последний (или первый — обычно одинаковые)
feats = feats.drop_duplicates(subset=['app_id', 'date_part'], keep='last').reset_index(drop=True)
print('feats после дедупа:', feats.shape)

# Снова: строковые object-колонки в feats — поищем и приведём
obj_cols_feats = feats.select_dtypes(include=['object']).columns.tolist()
print('object cols в feats:', len(obj_cols_feats))
print('пример:', obj_cols_feats[:10])

feats после дедупа: (76583, 280)
object cols в feats: 41
пример: ['app_id', 'date_part', 'clientsegment', 'country', 'clienttype', 'clientgroup', 'clientoutflowstatus', 'srvpackage_fst', 'clientsegment_prd', 'lp_client_group_name']


In [21]:
# Конвертация object-фичей в features
# Сначала попробуем как число, если не получилось — как категорию
for col in obj_cols_feats:
    if col in ('app_id', 'date_part'):
        continue
    # пробуем как число
    converted = pd.to_numeric(feats[col], errors='coerce')
    if converted.notna().sum() / max(feats[col].notna().sum(), 1) > 0.9:
        feats[col] = converted.astype('float32')
    else:
        # категория -> codes (общий маппинг для train/test не нужен, мы потом джойним)
        feats[col] = feats[col].fillna('NA').astype(str)
        feats[col] = pd.Categorical(feats[col]).codes.astype('int32')

# str-колонки тоже превратим в коды
str_cols = feats.select_dtypes(include=['object']).columns.tolist()
print('осталось object после конвертации:', str_cols)
# любые оставшиеся — в категориальные коды
for col in str_cols:
    if col in ('app_id', 'date_part'): continue
    feats[col] = pd.Categorical(feats[col].fillna('NA').astype(str)).codes.astype('int32')

print('dtypes counts:')
print(feats.dtypes.value_counts())

осталось object после конвертации: ['app_id', 'date_part']
dtypes counts:
float64    200
int32       39
float32     39
object       2
Name: count, dtype: int64


In [22]:
# === JOIN с features ===
JOIN_KEYS = ['app_id', 'date_part']

# Убираем колонки, которые могут конфликтовать (если в features есть свои)
feats_to_join = feats.drop(columns=[c for c in feats.columns 
                                     if c in train.columns and c not in JOIN_KEYS],
                            errors='ignore')

train = train.merge(feats_to_join, on=JOIN_KEYS, how='left')
test  = test.merge(feats_to_join,  on=JOIN_KEYS, how='left')

print('train после джойна:', train.shape)
print('test  после джойна:', test.shape)

# Проверка пропусков после джойна
miss_train = train.isna().sum().sum()
miss_test = test.isna().sum().sum()
print(f'NaN в train: {miss_train}, в test: {miss_test}')

train после джойна: (845110, 301)
test  после джойна: (936883, 300)
NaN в train: 75856065, в test: 83882194


In [23]:
# === LISTWISE FEATURE ENGINEERING (главный буст) ===
# Внутри request_id считаем ранги и отклонения по ключевым фичам оффера
LISTWISE_BASE = ['rate', 'eva', 'eva_perc', 'ncl', 'limit', 'term']

def add_listwise_features(df, group_col='request_id'):
    g = df.groupby(group_col)
    for col in LISTWISE_BASE:
        # ранг внутри заявки (нормированный)
        df[f'{col}_rank'] = g[col].rank(method='dense', pct=True).astype('float32')
        # z-score внутри заявки
        mean = g[col].transform('mean')
        std  = g[col].transform('std').replace(0, 1)
        df[f'{col}_z'] = ((df[col] - mean) / std).astype('float32')
        # отношение к минимуму/максимуму внутри заявки
        df[f'{col}_to_min'] = (df[col] / g[col].transform('min').replace(0, np.nan)).astype('float32')
        df[f'{col}_to_max'] = (df[col] / g[col].transform('max').replace(0, np.nan)).astype('float32')
    
    # длина списка вариантов в заявке
    df['n_variants_in_req'] = g[group_col].transform('size').astype('int32')
    return df

train = add_listwise_features(train)
test  = add_listwise_features(test)

# Дополнительно: соотношения с запросом клиента
for df in (train, test):
    df['limit_to_req'] = (df['limit'] / df['req_loan_amount'].replace(0, np.nan)).astype('float32')
    df['term_to_req']  = (df['term']  / df['req_term'].replace(0, np.nan)).astype('float32')

print('train после FE:', train.shape)
print('Listwise фичи добавлены')

train после FE: (845110, 328)
Listwise фичи добавлены


In [24]:
# === Финальный набор фичей ===
DROP_COLS = ['app_id', 'request_received', 'date_part', 'request_id', 
             'offer_id', 'is_deal', 'nb_date']  # nb_date может быть в feats

feature_cols = [c for c in train.columns 
                if c not in DROP_COLS and c in test.columns]

print('Финальное число фичей:', len(feature_cols))

# Категориальные для LightGBM
CAT_FEATURES = ['variant_no', 'verif_compl', 'offer_type', 'risk_level_map',
                'verif_need', 'need_2ndfl', 'channel', 'basket_name', 'pil1mtrx_offer']
CAT_FEATURES = [c for c in CAT_FEATURES if c in feature_cols]
print('cat features:', CAT_FEATURES)

Финальное число фичей: 321
cat features: ['variant_no', 'verif_compl', 'offer_type', 'risk_level_map', 'verif_need', 'need_2ndfl', 'channel', 'basket_name', 'pil1mtrx_offer']


In [25]:
# === Сортировка ОБЯЗАТЕЛЬНА для ranker'a ===
train = train.sort_values('request_id').reset_index(drop=True)
test  = test.sort_values('request_id').reset_index(drop=True)

X_train = train[feature_cols]
y_train = train['is_deal'].values
g_train = train['request_id'].values

X_test = test[feature_cols]

print('Готово к обучению')
print('X_train:', X_train.shape, 'y_train mean:', y_train.mean())

Готово к обучению
X_train: (845110, 321) y_train mean: 0.04483558353350452


In [26]:
# === GroupKFold + LightGBM lambdarank ===
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

oof = np.zeros(len(train))
test_pred_lgb = np.zeros(len(test))

models_lgb = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups=g_train)):
    print(f'\n===== Fold {fold} =====')
    
    tr_groups = train.iloc[tr_idx].groupby('request_id', sort=False).size().values
    va_groups = train.iloc[va_idx].groupby('request_id', sort=False).size().values
    
    dtr = lgb.Dataset(X_train.iloc[tr_idx], y_train[tr_idx], group=tr_groups,
                      categorical_feature=CAT_FEATURES)
    dva = lgb.Dataset(X_train.iloc[va_idx], y_train[va_idx], group=va_groups,
                      categorical_feature=CAT_FEATURES, reference=dtr)
    
    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'eval_at': [5],
        'lambdarank_truncation_level': 5,
        'learning_rate': 0.05,
        'num_leaves': 127,
        'min_data_in_leaf': 100,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'lambda_l2': 1.0,
        'verbose': -1,
        'seed': 42 + fold,
    }
    
    model = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(200)])
    
    oof[va_idx] = model.predict(X_train.iloc[va_idx])
    test_pred_lgb += model.predict(X_test) / N_SPLITS
    models_lgb.append(model)
    gc.collect()

print('\nLightGBM trained')


===== Fold 0 =====
Training until validation scores don't improve for 150 rounds
[200]	valid_0's ndcg@5: 0.9155
Early stopping, best iteration is:
[130]	valid_0's ndcg@5: 0.916762

===== Fold 1 =====
Training until validation scores don't improve for 150 rounds
[200]	valid_0's ndcg@5: 0.9074
Early stopping, best iteration is:
[172]	valid_0's ndcg@5: 0.908855

===== Fold 2 =====
Training until validation scores don't improve for 150 rounds
[200]	valid_0's ndcg@5: 0.912512
Early stopping, best iteration is:
[106]	valid_0's ndcg@5: 0.914365

===== Fold 3 =====
Training until validation scores don't improve for 150 rounds
[200]	valid_0's ndcg@5: 0.908285
Early stopping, best iteration is:
[104]	valid_0's ndcg@5: 0.910051

===== Fold 4 =====
Training until validation scores don't improve for 150 rounds
[200]	valid_0's ndcg@5: 0.912071
Early stopping, best iteration is:
[87]	valid_0's ndcg@5: 0.913187

LightGBM trained


In [27]:
# === Оценка OOF NDCG@5 ===
def ndcg_at_k_grouped(df, score_col='oof', target_col='is_deal', group_col='request_id', k=5):
    """NDCG@k вручную, корректно для one-positive setup."""
    scores = []
    for req, g in df.groupby(group_col, sort=False):
        if g[target_col].sum() == 0:
            continue
        # idcg = 1 (один позитив наверху)
        order = np.argsort(-g[score_col].values)
        ranks = np.where(g[target_col].values[order] == 1)[0]
        if len(ranks) == 0 or ranks[0] >= k:
            scores.append(0.0)
        else:
            # DCG = 1 / log2(rank+2), IDCG = 1 / log2(0+2) = 1
            scores.append(1.0 / np.log2(ranks[0] + 2))
    return np.mean(scores)

train['oof'] = oof
oof_score = ndcg_at_k_grouped(train, 'oof', 'is_deal', 'request_id', k=5)
print(f'OOF NDCG@5 (LightGBM lambdarank): {oof_score:.4f}')
print(f'Балл хакатона ≈ {oof_score * 100:.2f}')

OOF NDCG@5 (LightGBM lambdarank): 0.9126
Балл хакатона ≈ 91.26


In [28]:
# === Feature importance (топ-30) ===
imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': np.mean([m.feature_importance(importance_type='gain') for m in models_lgb], axis=0)
}).sort_values('importance', ascending=False)
print(imp.head(30))

                    feature     importance
5            pil1mtrx_offer  746391.596271
16              basket_name  159040.759918
306                ncl_rank  107311.938727
319            limit_to_req   84357.671371
320             term_to_req   83707.549825
307                   ncl_z   50104.784010
313            limit_to_max   49790.422923
3                variant_no   42947.147340
15                  channel   24299.529926
1                  req_term   19018.656947
4                offer_type   18091.519103
315                  term_z   13429.245705
310              limit_rank   11230.156863
311                 limit_z   11025.256348
309              ncl_to_max    6879.708727
318       n_variants_in_req    5805.994081
303              eva_perc_z    5663.162752
7                      term    5652.631861
70   socialparamchange_days    5558.610864
298                eva_rank    5483.100905
302           eva_perc_rank    4692.160202
295                  rate_z    4638.123258
305        

In [29]:
# === CatBoost YetiRank (вторая модель для ансамбля) ===
oof_cb = np.zeros(len(train))
test_pred_cb = np.zeros(len(test))

cb_cat_features = [feature_cols.index(c) for c in CAT_FEATURES if c in feature_cols]

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups=g_train)):
    print(f'\n===== CatBoost Fold {fold} =====')
    
    tr_data = X_train.iloc[tr_idx].copy()
    va_data = X_train.iloc[va_idx].copy()
    
    # CatBoost требует строковых cat-фичей или целочисленных без NaN
    for c in CAT_FEATURES:
        tr_data[c] = tr_data[c].fillna(-1).astype(int)
        va_data[c] = va_data[c].fillna(-1).astype(int)
    
    tr_pool = Pool(tr_data, y_train[tr_idx], 
                   group_id=g_train[tr_idx],
                   cat_features=cb_cat_features)
    va_pool = Pool(va_data, y_train[va_idx],
                   group_id=g_train[va_idx],
                   cat_features=cb_cat_features)
    
    model_cb = CatBoostRanker(
        loss_function='YetiRankPairwise',
        eval_metric='NDCG:top=5;type=Exp',
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        early_stopping_rounds=200,
        random_seed=42 + fold,
        verbose=200,
        task_type='GPU',  # если есть GPU; иначе убери эту строку
    )
    
    model_cb.fit(tr_pool, eval_set=va_pool)
    
    oof_cb[va_idx] = model_cb.predict(va_pool)
    
    test_data = X_test.copy()
    for c in CAT_FEATURES:
        test_data[c] = test_data[c].fillna(-1).astype(int)
    test_pred_cb += model_cb.predict(Pool(test_data, cat_features=cb_cat_features)) / N_SPLITS
    
    gc.collect()

train['oof_cb'] = oof_cb
print(f'\nOOF NDCG@5 (CatBoost YetiRank): {ndcg_at_k_grouped(train, "oof_cb"):.4f}')


===== CatBoost Fold 0 =====
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8701525	best: 0.8701525 (0)	total: 1.01s	remaining: 50m 43s
200:	test: 0.9131137	best: 0.9134009 (194)	total: 3m 23s	remaining: 47m 17s
400:	test: 0.9158530	best: 0.9160659 (379)	total: 6m 46s	remaining: 43m 55s
600:	test: 0.9168044	best: 0.9172228 (586)	total: 10m 9s	remaining: 40m 32s
bestTest = 0.9172228018
bestIteration = 586
Shrink model to first 587 iterations.

===== CatBoost Fold 1 =====
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8583809	best: 0.8583809 (0)	total: 1.02s	remaining: 51m 13s
200:	test: 0.9045355	best: 0.9047721 (197)	total: 3m 25s	remaining: 47m 38s
400:	test: 0.9057277	best: 0.9059316 (384)	total: 6m 48s	remaining: 44m 7s
600:	test: 0.9070640	best: 0.9072543 (594)	total: 10m 11s	remaining: 40m 41s
800:	test: 0.9079818	best: 0.9079966 (797)	total: 13m 34s	remaining: 37m 17s
1000:	test: 0.9078343	best: 0.9082128 (973)	total: 16m 58s	remaining: 33m 53s
1200:	test: 0.9081861	best: 0.9086260 (1105)	total: 20m 21s	remaining: 30m 30s
bestTest = 0.9086260104
bestIteration = 1105
Shrink model to first 1106 iterations.

===== CatBoost Fold 2 =====
Groupwise loss function. OneHotMaxSize set to 10


...

0:	test: 0.8632359	best: 0.8632359 (0)	total: 1.02s	remaining: 51m 7s
200:	test: 0.9097525	best: 0.9098942 (193)	total: 3m 24s	remaining: 47m 28s
400:	test: 0.9119490	best: 0.9123207 (347)	total: 6m 46s	remaining: 43m 57s
600:	test: 0.9131519	best: 0.9132222 (597)	total: 10m 9s	remaining: 40m 32s
800:	test: 0.9136570	best: 0.9139211 (775)	total: 13m 32s	remaining: 37m 9s
1000:	test: 0.9142135	best: 0.9143918 (991)	total: 16m 54s	remaining: 33m 45s
1200:	test: 0.9140760	best: 0.9145069 (1119)	total: 20m 17s	remaining: 30m 23s
1400:	test: 0.9141812	best: 0.9146726 (1337)	total: 23m 39s	remaining: 27m
1600:	test: 0.9146190	best: 0.9148243 (1562)	total: 27m 2s	remaining: 23m 37s
1800:	test: 0.9148994	best: 0.9151295 (1681)	total: 30m 24s	remaining: 20m 14s
2000:	test: 0.9147295	best: 0.9152446 (1930)	total: 33m 47s	remaining: 16m 52s
2200:	test: 0.9150490	best: 0.9153118 (2116)	total: 37m 10s	remaining: 13m 29s
bestTest = 0.9153117597
bestIteration = 2116
Shrink model to first 2117 iterati

Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8681896	best: 0.8681896 (0)	total: 1.02s	remaining: 51m 14s
200:	test: 0.9084441	best: 0.9084441 (200)	total: 3m 24s	remaining: 47m 30s
400:	test: 0.9100690	best: 0.9104355 (369)	total: 6m 47s	remaining: 44m
600:	test: 0.9109601	best: 0.9110606 (554)	total: 10m 9s	remaining: 40m 34s
800:	test: 0.9115962	best: 0.9115962 (800)	total: 13m 32s	remaining: 37m 10s
1000:	test: 0.9117608	best: 0.9119231 (997)	total: 16m 55s	remaining: 33m 47s
1200:	test: 0.9119821	best: 0.9121584 (1192)	total: 20m 17s	remaining: 30m 24s
1400:	test: 0.9122996	best: 0.9124416 (1373)	total: 23m 40s	remaining: 27m 1s
1600:	test: 0.9118834	best: 0.9126691 (1425)	total: 27m 3s	remaining: 23m 38s
bestTest = 0.9126690909
bestIteration = 1425
Shrink model to first 1426 iterations.

===== CatBoost Fold 4 =====
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8622247	best: 0.8622247 (0)	total: 1.02s	remaining: 50m 56s
200:	test: 0.9089967	best: 0.9091475 (194)	total: 3m 23s	remaining: 47m 19s
400:	test: 0.9101745	best: 0.9104415 (378)	total: 6m 46s	remaining: 43m 52s
600:	test: 0.9108616	best: 0.9110525 (590)	total: 10m 8s	remaining: 40m 28s
800:	test: 0.9111988	best: 0.9116279 (768)	total: 13m 30s	remaining: 37m 6s
1000:	test: 0.9114831	best: 0.9118755 (980)	total: 16m 53s	remaining: 33m 43s
1200:	test: 0.9119634	best: 0.9121560 (1123)	total: 20m 14s	remaining: 30m 19s
1400:	test: 0.9123435	best: 0.9126597 (1332)	total: 23m 37s	remaining: 26m 57s
bestTest = 0.9126597354
bestIteration = 1332
Shrink model to first 1333 iterations.

OOF NDCG@5 (CatBoost YetiRank): 0.9143


In [30]:
# === Ансамбль по рангам внутри request_id ===
# Rank-blending почти всегда стабильнее score-blending для NDCG

def rank_within_group(df, score_col, group_col='request_id'):
    return df.groupby(group_col)[score_col].rank(method='average', pct=True).values

train['oof_lgb_rank'] = rank_within_group(train, 'oof')
train['oof_cb_rank']  = rank_within_group(train, 'oof_cb')

# Подбор веса через бруте по OOF
best_w, best_score = 0.5, 0
for w in np.linspace(0, 1, 21):
    train['blend'] = w * train['oof_lgb_rank'] + (1 - w) * train['oof_cb_rank']
    s = ndcg_at_k_grouped(train, 'blend')
    if s > best_score:
        best_score, best_w = s, w

print(f'Лучший вес LGB: {best_w:.2f}, OOF NDCG@5 ансамбля: {best_score:.4f}')

# Применяем к тесту
test['pred_lgb'] = test_pred_lgb
test['pred_cb']  = test_pred_cb
test['rank_lgb'] = rank_within_group(test, 'pred_lgb')
test['rank_cb']  = rank_within_group(test, 'pred_cb')
test['score']    = best_w * test['rank_lgb'] + (1 - best_w) * test['rank_cb']

Лучший вес LGB: 0.45, OOF NDCG@5 ансамбля: 0.9153


In [31]:
# === SUBMISSION ===
submission = test[['request_id', 'variant_no', 'score']].copy()
submission.to_csv('submission.csv', index=False, sep=';')

print('Файл submission.csv:')
print(submission.head(10))
print(f'shape: {submission.shape}')

# Sanity-check
assert submission['score'].isna().sum() == 0, 'NaN в score!'
assert submission.shape[0] == len(test), 'Потеряли строки'
print('OK')

Файл submission.csv:
   request_id  variant_no  score
0           0          10    0.1
1           0           1    1.0
2           0           6    0.5
3           0           9    0.2
4           0           4    0.8
5           0           3    0.9
6           0           8    0.3
7           0           2    0.6
8           0           5    0.7
9           0           7    0.4
shape: (936883, 3)
OK
